# 07 · Gold — physical_itens_venda_caixa
**Projeto:** Estágio Engenharia de Dados · **Squad:** 3 — Batch · **Autor:** Alexandro Silva

Terceira camada da arquitetura Medallion. Lê a Silver, gera as tabelas sumarizadas dos KPIs e grava
cada uma em dois destinos: Delta na camada Gold do ADLS e tabela no SQL Server, prontas para o Looker.

## KPIs gerados por este notebook
Regras de negócio 6 a 10 de `physical_itens_venda_caixa` (planilha de regras da Squad 3):

| Nº | KPI | Tabela Gold | Granularidade (1 linha por) |
|---|---|---|---|
| 6 | Receita por produto por loja por mês | `gold_itens_caixa_receita_produto_loja_mes` | produto, loja, mês |
| 7 | Top 10 produtos mais vendidos (quantidade e receita) por loja por trimestre | `gold_itens_caixa_top10_produto_loja_trimestre` | produto, loja, trimestre |
| 8 | Crescimento MoM da receita de perecíveis vs secos por loja | `gold_itens_caixa_mom_tipo_produto_loja` | tipo de produto, loja, mês |
| 9 | Quantidade média de itens por transação por loja | `gold_itens_caixa_itens_por_transacao_loja_mes` | loja, mês |
| 10 | Enriquecimento: flag `venda_em_feriado` (feriados nacionais) | `gold_itens_caixa_feriado_loja_ano` | loja, ano, feriado ou não |

## Origem e destino
| Item | Definição |
|---|---|
| Origem | `abfss://squad3@internshipdatalake.dfs.core.windows.net/silver/physical_itens_venda_caixa` |
| Destino Delta | `abfss://squad3@internshipdatalake.dfs.core.windows.net/gold/<tabela>` |
| Destino SQL Server | `squad3.<tabela>` |
| Modo | upsert no Delta (equivalente ao `MERGE`; ver etapa 4); o SQL Server recebe o conteúdo do Delta |

## Etapas
1. Credenciais e caminhos
2. Leitura da Silver e das tabelas de apoio
3. Base enriquecida (loja, data, tipo de produto e feriado)
4. Função de gravação (Delta + SQL Server)
5. KPI 6 — Receita por produto por loja por mês
6. KPI 7 — Top 10 produtos por loja por trimestre
7. KPI 8 — Crescimento MoM de perecíveis vs secos por loja
8. KPI 9 — Itens por transação por loja
9. KPI 10 — Vendas em feriado
10. Validação

## 1. Credenciais e caminhos
Carrega do `.env` as credenciais do ADLS e do SQL Server. As tabelas de apoio (vendas, lojas e catálogo
de perecíveis) vêm **provisoriamente** das cópias derivadas da RAW; devem passar a vir da Silver das
duplas responsáveis assim que forem publicadas.

In [0]:
from dotenv import load_dotenv
import os

load_dotenv("/Workspace/Users/alexandrofs31@gmail.com/.env")

CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
TENANT_ID       = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT", "internshipdatalake")
SQL_HOST        = os.getenv("SQL_HOST")
SQL_DATABASE    = os.getenv("SQL_DATABASE")
SQL_USERNAME    = os.getenv("SQL_USERNAME")
SQL_PASSWORD    = os.getenv("SQL_PASSWORD")

obrigatorias = {"ADLS_CLIENT_ID": CLIENT_ID, "ADLS_TENANT_ID": TENANT_ID, "ADLS_CLIENT_SECRET": CLIENT_SECRET,
                "SQL_HOST": SQL_HOST, "SQL_DATABASE": SQL_DATABASE, "SQL_USERNAME": SQL_USERNAME,
                "SQL_PASSWORD": SQL_PASSWORD}
faltando = [n for n, v in obrigatorias.items() if not v]
if faltando:
    raise ValueError(f"Variáveis ausentes no .env: {faltando}")

TABELA         = "physical_itens_venda_caixa"
BASE_SQUAD     = f"abfss://squad3@{STORAGE_ACCOUNT}.dfs.core.windows.net"
CAMINHO_SILVER = f"{BASE_SQUAD}/silver/{TABELA}"
CAMINHO_GOLD   = f"{BASE_SQUAD}/gold"
SCHEMA_SQL     = "squad3"

# PROVISÓRIO: cópias derivadas da RAW. Trocar pela Silver das duplas responsáveis.
TABELA_VENDAS_REF     = "workspace.squad3.physical_vendas_caixa"
TABELA_LOJAS_REF      = "workspace.squad3.physical_lojas"
TABELA_PERECIVEIS_REF = "workspace.squad3.physical_produtos_pereciveis"

# True = recria as tabelas Gold do zero (use UMA vez, quando o schema mudar). Depois volte para False.
RECRIAR_GOLD = False

sufixo = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
opcoes_adls = {
    f"fs.azure.account.auth.type.{sufixo}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{sufixo}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{sufixo}": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{sufixo}": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{sufixo}": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

print(f"📂 Silver     : {CAMINHO_SILVER}")
print(f"📦 Gold Delta : {CAMINHO_GOLD}/<tabela>")
print(f"🗄️  SQL Server : {SQL_DATABASE} · schema {SCHEMA_SQL}")

## 2. Leitura da Silver e das tabelas de apoio
A tabela de itens não tem loja nem data: as duas vêm de `physical_vendas_caixa`, pelo `id_transacao`.
O nome da loja vem de `physical_lojas`, e o catálogo de perecíveis define o tipo de produto do KPI 8.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date, timedelta

df_silver = spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_SILVER)
total_silver = df_silver.count()
receita_silver = df_silver.agg(F.sum("valor_total_item")).collect()[0][0]

df_vendas = (spark.table(TABELA_VENDAS_REF)
    .select(F.trim(F.col("id_transacao")).alias("id_transacao"),
            F.col("id_loja").cast("int").alias("id_loja"),
            F.to_timestamp("dt_venda").alias("dt_venda")))

df_lojas = (spark.table(TABELA_LOJAS_REF)
    .select(F.col("id_loja").cast("int").alias("id_loja"), "nome_loja", "cidade_loja", "estado_loja"))

df_pereciveis = (spark.table(TABELA_PERECIVEIS_REF)
    .select(F.lower(F.trim(F.col("sku"))).alias("codigo_barras_produto"))
    .dropDuplicates()
    .withColumn("_perecivel", F.lit(True)))

print(f"✅ Silver : {total_silver:,} itens | receita R$ {receita_silver:,.2f}")
print(f"✅ Vendas : {df_vendas.count():,} transações")
print(f"✅ Lojas  : {df_lojas.count():,}")
print(f"✅ Catálogo de perecíveis: {df_pereciveis.count():,} produtos")

## 3. Base enriquecida
Junta a cada item a loja, a data da venda, o tipo de produto (perecível ou seco) e a flag
`venda_em_feriado`, que é o enriquecimento da regra 10. Os feriados nacionais são gerados no próprio
notebook: os de data fixa mais a Sexta-feira Santa, calculada a partir da Páscoa de cada ano.

A célula também reporta inconsistências que afetam os KPIs: itens sem loja ou sem data e lojas que
aparecem nas vendas mas não existem no cadastro.

In [0]:
def pascoa(ano):
    """Domingo de Páscoa pelo algoritmo gregoriano anônimo."""
    a, b, c = ano % 19, ano // 100, ano % 100
    d, e = b // 4, b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = c // 4, c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    mes = (h + l - 7 * m + 114) // 31
    dia = (h + l - 7 * m + 114) % 31 + 1
    return date(ano, mes, dia)

FERIADOS_FIXOS = {(1, 1): "Confraternização Universal", (4, 21): "Tiradentes", (5, 1): "Dia do Trabalho",
                  (9, 7): "Independência", (10, 12): "Nossa Senhora Aparecida", (11, 2): "Finados",
                  (11, 15): "Proclamação da República", (11, 20): "Consciência Negra", (12, 25): "Natal"}

limites = df_vendas.agg(F.min("dt_venda"), F.max("dt_venda")).collect()[0]
anos = range(limites[0].year, limites[1].year + 1)
feriados = []
for ano in anos:
    feriados += [(date(ano, m, d), nome) for (m, d), nome in FERIADOS_FIXOS.items()]
    feriados.append((pascoa(ano) - timedelta(days=2), "Sexta-feira Santa"))
df_feriados = spark.createDataFrame(feriados, "data_venda date, nome_feriado string")

df_base = (df_silver
    .select("id_item_venda", "id_transacao", "codigo_barras_produto", "unidade_medida",
            "quantidade", "valor_total_item")
    .join(df_vendas, on="id_transacao", how="left")
    .join(F.broadcast(df_lojas), on="id_loja", how="left")
    .join(F.broadcast(df_pereciveis), on="codigo_barras_produto", how="left")
    .withColumn("tipo_produto", F.when(F.col("_perecivel"), "perecivel").otherwise("seco"))
    .withColumn("data_venda", F.to_date("dt_venda"))
    .join(F.broadcast(df_feriados), on="data_venda", how="left")
    .withColumn("venda_em_feriado", F.col("nome_feriado").isNotNull())
    .withColumn("ano", F.year("dt_venda"))
    .withColumn("mes", F.month("dt_venda"))
    .withColumn("trimestre", F.quarter("dt_venda"))
    .drop("_perecivel"))

c = df_base.agg(
    F.sum(F.col("id_loja").isNull().cast("int")).alias("itens_sem_loja"),
    F.sum(F.col("dt_venda").isNull().cast("int")).alias("itens_sem_data"),
    F.sum((F.col("id_loja").isNotNull() & F.col("nome_loja").isNull()).cast("int")).alias("itens_de_loja_fora_do_cadastro"),
    F.sum(F.col("venda_em_feriado").cast("int")).alias("itens_vendidos_em_feriado"),
    F.sum((F.col("tipo_produto") == "perecivel").cast("int")).alias("itens_pereciveis"),
).collect()[0].asDict()

print(f"📅 Feriados nacionais gerados: {len(feriados)} ({anos.start} a {anos.stop - 1})")
for nome, qtd in c.items():
    qtd = qtd or 0
    alerta = "⚠️ " if nome.startswith("itens_sem") or "fora_do_cadastro" in nome else "ℹ️ "
    if alerta == "⚠️ " and qtd == 0:
        alerta = "✅"
    print(f"{alerta} {nome:<32} {qtd:>10,} ({qtd / total_silver * 100:6.2f}%)")

lojas_fora = (df_base.filter(F.col("id_loja").isNotNull() & F.col("nome_loja").isNull())
    .select("id_loja").distinct().orderBy("id_loja").collect())
if lojas_fora:
    print(f"\n⚠️  Lojas com vendas e sem cadastro em physical_lojas: {[r.id_loja for r in lojas_fora]}")
    print("    Informar ao professor: essas lojas aparecem nos KPIs sem nome, cidade e estado.")

## 4. Função de gravação (Delta + SQL Server)
Cada tabela Gold é gravada em dois destinos:

- **Delta na camada Gold**, com upsert pelas chaves da tabela: chave existente é atualizada, chave nova
  é inserida e a que não veio é mantida. O comando `MERGE` do Delta não funciona neste ambiente
  (`AZURE_INVALID_CREDENTIALS_CONFIGURATION` e `CONFIG_NOT_AVAILABLE` no Serverless), então o upsert é
  feito com leitura e gravação, como na Silver.
- **SQL Server**, em `squad3.<tabela>`, recebendo o conteúdo completo do Delta já atualizado, para que
  os dois destinos fiquem sempre idênticos.

No ranking do top 10 a chave é o grupo (loja e trimestre), e não a linha: o grupo é substituído por
inteiro, para que um produto que saiu do top 10 não continue na tabela.

In [0]:
resumo_gravacao = []

def gravar_gold(df, nome, chaves, particao=None):
    """Grava uma tabela Gold em Delta (upsert por `chaves`) e no SQL Server."""
    caminho = f"{CAMINHO_GOLD}/{nome}"
    df = df.withColumn("gold_processed_at", F.current_timestamp())

    try:
        df_atual = spark.read.format("delta").options(**opcoes_adls).load(caminho)
        df_atual.limit(1).collect()
        existe = True
    except Exception:
        existe = False

    if RECRIAR_GOLD or not existe:
        df_final = df
    else:
        df_mantidas = df_atual.join(df.select(*chaves).distinct(), on=chaves, how="left_anti")
        df_final = df.unionByName(df_mantidas)

    gravador = (df_final.write.format("delta").options(**opcoes_adls).mode("overwrite")
        .option("overwriteSchema", "true" if RECRIAR_GOLD else "false"))
    if particao:
        gravador = gravador.partitionBy(*particao)
    gravador.save(caminho)

    df_gold = spark.read.format("delta").options(**opcoes_adls).load(caminho)
    (df_gold.write
        .format("sqlserver")
        .option("host", SQL_HOST)
        .option("port", "1433")
        .option("database", SQL_DATABASE)
        .option("dbtable", f"{SCHEMA_SQL}.{nome}")
        .option("user", SQL_USERNAME)
        .option("password", SQL_PASSWORD)
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .mode("overwrite")
        .save())

    linhas = df_gold.count()
    resumo_gravacao.append((nome, linhas))
    print(f"✅ {nome}: {linhas:,} linhas | Delta: gold/{nome} | SQL Server: {SCHEMA_SQL}.{nome}")
    return df_gold

## 5. KPI 6 — Receita por produto por loja por mês
Sell-out da loja física: receita, quantidade vendida e número de itens de cada produto, por loja e mês.
É a maior tabela Gold e fica particionada por `ano` e `mes`.

In [0]:
df_kpi6 = (df_base
    .groupBy("ano", "mes", "id_loja", "nome_loja", "cidade_loja", "estado_loja",
             "codigo_barras_produto", "tipo_produto", "unidade_medida")
    .agg(F.sum("valor_total_item").alias("receita"),
         F.sum("quantidade").alias("quantidade_vendida"),
         F.count("*").alias("qtd_itens")))

gold_kpi6 = gravar_gold(df_kpi6, "gold_itens_caixa_receita_produto_loja_mes",
                        chaves=["ano", "mes", "id_loja", "codigo_barras_produto"],
                        particao=["ano", "mes"])

## 6. KPI 7 — Top 10 produtos por loja por trimestre
Dois rankings por loja e trimestre: por receita e por quantidade vendida. Ficam na tabela os produtos
que estão entre os 10 primeiros em pelo menos um dos dois; as colunas `rank_receita` e
`rank_quantidade` permitem filtrar cada ranking no Looker.

In [0]:
df_trimestre = (df_base
    .groupBy("ano", "trimestre", "id_loja", "nome_loja", "codigo_barras_produto", "tipo_produto")
    .agg(F.sum("valor_total_item").alias("receita"),
         F.sum("quantidade").alias("quantidade_vendida")))

grupo = Window.partitionBy("ano", "trimestre", "id_loja")
df_kpi7 = (df_trimestre
    .withColumn("rank_receita", F.row_number().over(
        grupo.orderBy(F.col("receita").desc(), F.col("codigo_barras_produto"))))
    .withColumn("rank_quantidade", F.row_number().over(
        grupo.orderBy(F.col("quantidade_vendida").desc(), F.col("codigo_barras_produto"))))
    .filter((F.col("rank_receita") <= 10) | (F.col("rank_quantidade") <= 10)))

# chave = grupo (loja + trimestre): o ranking do grupo é substituído por inteiro
gold_kpi7 = gravar_gold(df_kpi7, "gold_itens_caixa_top10_produto_loja_trimestre",
                        chaves=["ano", "trimestre", "id_loja"])

## 7. KPI 8 — Crescimento MoM de perecíveis vs secos por loja
Receita mensal por loja e tipo de produto, com a receita do mês anterior e o crescimento percentual:
`(receita − receita_mes_anterior) / receita_mes_anterior × 100`. O primeiro mês de cada loja fica sem
crescimento, e o mês anterior só é usado quando é de fato o mês imediatamente anterior.

In [0]:
df_mensal = (df_base
    .groupBy("ano", "mes", "id_loja", "nome_loja", "tipo_produto")
    .agg(F.sum("valor_total_item").alias("receita"))
    .withColumn("_ordem", F.col("ano") * 12 + F.col("mes")))

serie = Window.partitionBy("id_loja", "tipo_produto").orderBy("_ordem")
df_kpi8 = (df_mensal
    .withColumn("_ordem_anterior", F.lag("_ordem").over(serie))
    .withColumn("receita_mes_anterior",
        F.when(F.col("_ordem") - F.col("_ordem_anterior") == 1, F.lag("receita").over(serie)))
    .withColumn("crescimento_mom_pct",
        F.round((F.col("receita") - F.col("receita_mes_anterior"))
                / F.col("receita_mes_anterior") * 100, 2).cast("decimal(12,2)"))
    .drop("_ordem", "_ordem_anterior"))

gold_kpi8 = gravar_gold(df_kpi8, "gold_itens_caixa_mom_tipo_produto_loja",
                        chaves=["ano", "mes", "id_loja", "tipo_produto"])

## 8. KPI 9 — Itens por transação por loja
Ticket em itens: número de linhas de item dividido pelo número de transações, por loja e mês. A tabela
leva também as duas contagens, para o Looker recalcular a média em qualquer agregação (por loja no
período todo, por exemplo) sem fazer média de médias.

In [0]:
df_kpi9 = (df_base
    .groupBy("ano", "mes", "id_loja", "nome_loja", "cidade_loja", "estado_loja")
    .agg(F.countDistinct("id_transacao").alias("qtd_transacoes"),
         F.count("*").alias("qtd_itens"),
         F.sum("valor_total_item").alias("receita"))
    .withColumn("media_itens_por_transacao",
        F.round(F.col("qtd_itens") / F.col("qtd_transacoes"), 2).cast("decimal(10,2)")))

gold_kpi9 = gravar_gold(df_kpi9, "gold_itens_caixa_itens_por_transacao_loja_mes",
                        chaves=["ano", "mes", "id_loja"])

## 9. KPI 10 — Vendas em feriado
Compara dias de feriado nacional com dias normais, por loja e ano: quantos dias tiveram venda, quantos
itens foram vendidos e a média de itens por dia. A média por dia é a medida que permite comparar,
porque há muito menos feriados do que dias normais.

In [0]:
df_kpi10 = (df_base
    .groupBy("ano", "id_loja", "nome_loja", "venda_em_feriado")
    .agg(F.countDistinct("data_venda").alias("qtd_dias_com_venda"),
         F.countDistinct("id_transacao").alias("qtd_transacoes"),
         F.count("*").alias("qtd_itens"),
         F.sum("valor_total_item").alias("receita"))
    .withColumn("media_itens_por_dia",
        F.round(F.col("qtd_itens") / F.col("qtd_dias_com_venda"), 2).cast("decimal(12,2)")))

gold_kpi10 = gravar_gold(df_kpi10, "gold_itens_caixa_feriado_loja_ano",
                         chaves=["ano", "id_loja", "venda_em_feriado"])

## 10. Validação
Confere se os totais das tabelas Gold fecham com a Silver: a receita somada em cada tabela agregada
deve ser igual à receita da Silver, e o número de itens também. Em seguida relê cada tabela do SQL
Server e compara a contagem de linhas com o Delta.

In [0]:
print("=" * 78)
print("RECONCILIAÇÃO COM A SILVER")
print("=" * 78)
print(f"Silver: {total_silver:,} itens | receita R$ {receita_silver:,.2f}\n")

conferencias = [("KPI 6  produto/loja/mês", gold_kpi6), ("KPI 8  MoM tipo/loja", gold_kpi8),
                ("KPI 9  itens/transação", gold_kpi9), ("KPI 10 feriado", gold_kpi10)]
divergencias = []
for nome, df in conferencias:
    receita = df.agg(F.sum("receita")).collect()[0][0]
    ok = receita == receita_silver
    if not ok:
        divergencias.append(nome)
    print(f"{'✅' if ok else '❌'} {nome:<26} receita R$ {receita:,.2f}")

itens_kpi6 = gold_kpi6.agg(F.sum("qtd_itens")).collect()[0][0]
print(f"{'✅' if itens_kpi6 == total_silver else '❌'} KPI 6  itens somados        {itens_kpi6:,}")
if itens_kpi6 != total_silver:
    divergencias.append("KPI 6 itens")

print(f"ℹ️  KPI 7  top 10: {gold_kpi7.count():,} linhas em "
      f"{gold_kpi7.select('ano', 'trimestre', 'id_loja').distinct().count():,} grupos (loja × trimestre)")

print("\n" + "=" * 78)
print("DELTA × SQL SERVER")
print("=" * 78)
for nome, linhas_delta in resumo_gravacao:
    linhas_sql = (spark.read.format("sqlserver")
        .option("host", SQL_HOST).option("port", "1433").option("database", SQL_DATABASE)
        .option("dbtable", f"{SCHEMA_SQL}.{nome}")
        .option("user", SQL_USERNAME).option("password", SQL_PASSWORD)
        .option("encrypt", "true").option("trustServerCertificate", "false")
        .load().count())
    ok = linhas_sql == linhas_delta
    if not ok:
        divergencias.append(f"SQL {nome}")
    print(f"{'✅' if ok else '❌'} {nome:<50} Delta {linhas_delta:>9,} | SQL {linhas_sql:>9,}")

if divergencias:
    raise ValueError(f"Divergências na validação da Gold: {divergencias}")
print("\n✅ Gold validada: totais iguais aos da Silver e SQL Server igual ao Delta.")